# 线性代数进阶：数据表示与空间变换

在机器学习中，数据被表示为张量，而模型的操作（如特征提取）本质上是空间变换。

---

## 1. 张量 (Tensor) 的运算与重塑

### 定义
张量是多维数组。在深度学习框架中，张量的重塑 (Reshape)、转置 (Transpose) 和张量积 (Tensor Product) 是最常见的操作。

### 生动比喻
重塑张量就像是**玩乐高积木**。你有一定数量的积木（元素），你可以把它们拼成一排（向量），也可以叠成一个立方体（3D张量），只要积木的总数不变，形状可以随意变换。

### 手写实现：张量重塑与转置

In [ ]:
import numpy as np

def manual_reshape(arr, new_shape):
    """手写简单的 1D 转 2D reshape"""
    flat = [item for row in arr for item in row] if isinstance(arr[0], list) else arr
    rows, cols = new_shape
    return [flat[i*cols : (i+1)*cols] for i in range(rows)]

data = [1, 2, 3, 4, 5, 6]
print("重塑结果 (2x3):", manual_reshape(data, (2, 3)))

# Numpy 库函数调用
tensor_3d = np.arange(24).reshape(2, 3, 4)
print("Numpy 3D 张量形状:", tensor_3d.shape)
print("转置后的形状 (0, 2, 1):", tensor_3d.transpose(0, 2, 1).shape)

## 2. 协方差矩阵 (Covariance Matrix)

### 定义
给定 $n$ 个样本、每个样本 $d$ 维。先对数据做去中心化，记均值向量为 $\mu$，则协方差矩阵定义为：
$$C = \frac{1}{n-1}(X - \mu)^T(X - \mu)$$
其中，$C$ 的对角线元素表示各特征的方差，非对角线元素表示两个特征之间的协方差。

### 小数据实例：二维样本的手算
设有 3 个二维样本：
$$X = \begin{bmatrix} 1 & 2 \\ 2 & 4 \\ 3 & 6 \end{bmatrix}$$
它们的均值为：
$$\mu = \left[\frac{1+2+3}{3}, \frac{2+4+6}{3}\right] = [2, 4]$$
先去中心化：
$$X_c = X - \mu = \begin{bmatrix} -1 & -2 \\ 0 & 0 \\ 1 & 2 \end{bmatrix}$$
代入协方差公式：
$$C = \frac{1}{3-1} X_c^T X_c = \frac{1}{2} \begin{bmatrix} (-1)^2 + 0^2 + 1^2 & (-1)(-2) + 0\cdot0 + 1\cdot2 \\ (-2)(-1) + 0\cdot0 + 2\cdot1 & (-2)^2 + 0^2 + 2^2 \end{bmatrix}$$
$$C = \frac{1}{2} \begin{bmatrix} 2 & 4 \\ 4 & 8 \end{bmatrix} = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$$
这个例子里，第二个特征正好是第一个特征的 2 倍，所以协方差矩阵中的相关性非常强。

### 库函数调用
NumPy 里可以直接用 `np.cov` 计算协方差矩阵，和手算结果一致。

### 过渡到特征分解
协方差矩阵通常是对称矩阵，后续可以继续对它做特征分解，得到主方向。

In [ ]:
import numpy as np

# 小数据实例：二维样本，便于手算与对照
small_data = np.array([
    [1.0, 2.0],
    [2.0, 4.0],
    [3.0, 6.0],
])

# NumPy 库函数调用
cov_matrix = np.cov(small_data, rowvar=False)

print("小数据样本:\n", small_data)
print("NumPy 协方差矩阵:\n", cov_matrix)

## 3. 特征分解 (Eigen Decomposition)

### 数学原理
对于实对称矩阵 $A$，它可以分解为：
$$A = Q \Lambda Q^T$$
其中 $Q$ 是由特征向量组成的单位正交矩阵，$\Lambda$ 是对角矩阵，对角线上是特征值。

### 小数据案例：特征值与特征向量手算推导
考虑矩阵 $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$。我们要找到它的特征值 $\lambda$ 和特征向量 $v$。

**第一步：列出特征方程**
根据定义 $Av = \lambda v$，变形得 $(A - \lambda I)v = 0$。要使 $v$ 有非零解，必须满足行列式为 0：
$$\det(A - \lambda I) = 0$$
$$\det\left(\begin{bmatrix} 4-\lambda & 1 \\ 2 & 3-\lambda \end{bmatrix}\right) = 0$$

**第二步：解特征多项式**
$(4-\lambda)(3-\lambda) - (1 \times 2) = 0$
$\lambda^2 - 7\lambda + 12 - 2 = 0$
$\lambda^2 - 7\lambda + 10 = 0$
因式分解得：$(\lambda - 5)(\lambda - 2) = 0$
所以特征值为：$\lambda_1 = 5, \lambda_2 = 2$

**第三步：求特征向量**
- 当 $\lambda_1 = 5$ 时：
$\begin{bmatrix} 4-5 & 1 \\ 2 & 3-5 \end{bmatrix} \begin{bmatrix} v_1 \\ v_2 \end{bmatrix} = \begin{bmatrix} -1 & 1 \\ 2 & -2 \end{bmatrix} \begin{bmatrix} v_1 \\ v_2 \end{bmatrix} = 0$
得到方程 $-v_1 + v_2 = 0 \Rightarrow v_1 = v_2$。取特征向量 $v_1 = [1, 1]^T$。

- 当 $\lambda_2 = 2$ 时：
$\begin{bmatrix} 4-2 & 1 \\ 2 & 3-2 \end{bmatrix} \begin{bmatrix} v_1 \\ v_2 \end{bmatrix} = \begin{bmatrix} 2 & 1 \\ 2 & 1 \end{bmatrix} \begin{bmatrix} v_1 \\ v_2 \end{bmatrix} = 0$
得到方程 $2v_1 + v_2 = 0 \Rightarrow v_2 = -2v_1$。取特征向量 $v_2 = [1, -2]^T$。

### 手写实现：简化版 PCA (主成分分析)

In [ ]:
import matplotlib.pyplot as plt

# 1. 创建简单的小规模数据集 (2D)
X = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0], [2.3, 2.7], [2, 1.6], [1, 1.1], [1.5, 1.6], [1.1, 0.9]])

# 2. 去中心化 (Zero-centered)
X_mean = np.mean(X, axis=0)
X_centered = X - X_mean

# 3. 计算协方差矩阵 C = (X^T * X) / (n-1)
cov_matrix = np.cov(X_centered.T)

# 4. 计算特征值和特征向量
values, vectors = np.linalg.eig(cov_matrix)

# 5. 可视化主成分方向
plt.figure(figsize=(8, 6))
plt.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, label='Centered Data')
for i in range(len(values)):
    # 绘制特征向量方向，长度由特征值决定
    start, end = [0, 0], vectors[:, i] * np.sqrt(values[i]) * 2
    plt.annotate('', xy=end, xytext=start, arrowprops=dict(facecolor='red', width=2))
    plt.text(end[0], end[1], f'PC{i+1} (val={values[i]:.2f})', color='red')

plt.axis('equal')
plt.title("Principal Component Analysis (PCA) with Eigenvectors")
plt.grid(True)
plt.show()

## 3. 库函数实现：Sklearn PCA 对比

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(X)
print("Sklearn 特征值 (方差解释量):", pca.explained_variance_)
print("Sklearn 主成分方向:\n", pca.components_)

### 练习与思考
1. **计算题**：求矩阵 $A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}$ 的特征值。
2. **应用题**：在图像压缩中，特征分解是如何帮助我们减少存储空间的？
3. **进阶题**：奇异值分解 (SVD) 与特征分解有什么区别？为什么 SVD 在处理非方阵时更通用？